In [1]:

import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))
sys.path.insert(0, os.path.abspath("../sdlarch-rl"))


from pathlib import Path
import os
import re
from utils import get_last_index
import math
from mario import make_env
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import SubprocVecEnv
import numpy as np
import json

env = SubprocVecEnv(
    [make_env(11, human=False)]
)

# --- CONFIGS ---
MAX_TRAJ = 0
EVAL_JSON = "eval.json"
NUMBER_OF_EVALS=1


# ppo config
train_path = "./recurrent_ppo/"
sub_train_path = train_path + "steps"
last_index_imitation = get_last_index(str(sub_train_path), "ppo_policy", "zip")
current_epoch = last_index_imitation

MAX_TRAJ = last_index_imitation


#TODO comment
current_epoch = 60

file_path = Path(EVAL_JSON)
if not file_path.is_file():
    print("Creating initial json:")
    with open(EVAL_JSON, "w") as f:
        data = {
            "better_time": 0,
            "better_time_epoch": 999999999,
            "better_epoch": -999999999,
            "better_epoch_value": -999999999,
            "current_epoch": 60
        }
        json.dump(data, f, indent=4)


def make_policy(current_epoch):
    latest_model_path = sub_train_path + f"/ppo_policy{current_epoch}.zip"
    policy = RecurrentPPO.load(latest_model_path)
    

    return policy

count_record  = 0

with open(EVAL_JSON, 'r') as f:
    data = json.load(f)
    print(f"Loaded json by last training: {data}")

better_epoch=data["better_epoch"]
better_time=data["better_time"]
current_epoch=data["current_epoch"]
better_time_epoch = data["better_time_epoch"]
better_epoch_value = data["better_epoch_value"]


while count_record < MAX_TRAJ:
    policy = make_policy(current_epoch)
    

    total_reward = 0
    count_ended = 0
    total_time = 0

    print(f"Eval epoch: {current_epoch}")
    
    for episode in range(NUMBER_OF_EVALS):
        obs = env.reset()
        done = False

        lstm_states = None
        episode_starts = np.ones((1,), dtype=bool)
        info = None
        
        while not done:
            pred_act, lstm_states = policy.predict(obs, state=lstm_states, episode_start=episode_starts, deterministic=True)
            obs, reward, terminated, info = env.step(pred_act)
            done = terminated[0]
            episode_starts = terminated
            total_reward += reward[0]

        x = info[0].get("x", 0)
        if x >= 6681.0:
            current_time = info[0].get("time", 0)
            total_time += current_time
            count_ended += 1
            print(f"Current time, count_ended: {current_time}, {count_ended}")
            print(f"Total time current: {total_time/count_ended}")
        print(f"Total Acc Reward: {total_reward}")

    if count_ended > 0:
        total_time = total_time/count_end
        print(f"Total time: {total_time}")

    print(f"Total Reward: {total_reward}")
    total_reward = total_reward/NUMBER_OF_EVALS
    

    if total_reward > better_epoch_value:
        better_epoch = current_epoch
        better_epoch_value = total_reward
        print(f"Better Epoch: {current_epoch}")

    if total_time > better_time:
        better_time = total_time
        better_time_epoch = current_epoch
        print(f"Better time by epoch: {current_epoch}")

    print()

    current_epoch += 1

    with open(EVAL_JSON, "w") as f:
        data = {
            "better_time":better_time,
            "better_epoch": better_epoch,
            "current_epoch": current_epoch,
            "better_time_epoch": better_time_epoch,
            "better_epoch_value": better_epoch_value,
        }
        json.dump(data, f, indent=4)
        
    if current_epoch > last_index_imitation:
        break
    


D:\anaconda3\envs\env311\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Loaded json by last training: {'better_time': 443.0, 'better_epoch': 88, 'current_epoch': 179, 'better_time_epoch': 157, 'better_epoch_value': 32.33690275493409}


D:\anaconda3\envs\env311\Lib\site-packages\stable_baselines3\common\utils.py:166: UserWarning: get_schedule_fn() is deprecated, please use FloatSchedule() instead
  warnings.warn("get_schedule_fn() is deprecated, please use FloatSchedule() instead")


Eval epoch: 179
Current time: 432.0, 1
Total time current: 432.0
Total Acc Reward: 31.572145353618435
Total time: 432.0
Total Reward: 31.572145353618435

